
# ECG Explainability with SHAP & LIME (Overlay Visuals)

This notebook adds explainable AI (XAI) for your ECG classifier using **SHAP** and **LIME** and overlays the contributions directly on the ECG traces. It is designed to plug into your existing LSTM/Transformer/1D-CNN Keras model.

**What you'll get:**
- SHAP explanations (per-timestep, per-lead) for a chosen class (e.g., *Atrial Fibrillation* vs *Normal*).
- LIME explanations via 1D segmentation/perturbation.
- Publication-ready overlay plots (red = pushes toward target class, blue = pushes away).
- Text summaries of suspicious segments and leads.

> Tip: If you're starting from your existing `01_lstm_baseline-xai.ipynb`, you can run this notebook side-by-side or import the helper functions here.


In [ ]:

# %% [markdown]
# ## 0) Setup
# Install packages if needed (uncomment if running locally).
# Note: If you're in an offline environment, install these beforehand.
# !pip install shap lime scikit-learn numpy matplotlib tensorflow==2.*
#
# Optional: accelerate SHAP for TF2
# !pip install numba


In [ ]:

# %%
# 1) Config — set your paths and target class here
MODEL_PATH = "models/lstm_baseline.h5"   # <-- change to your model file
DATA_PATH  = "data/ecg_test.npy"         # numpy array shaped (N, T, L) or (N, T) for single lead
LABELS_PATH = "data/y_test.npy"          # numpy array shaped (N,) or (N, num_classes)
CLASS_NAMES = ['Normal', 'AF', 'Other']  # <-- edit to match your model
TARGET_CLASS_NAME = 'AF'                 # class you want to explain (positive class)
BACKGROUND_N = 200                       # background size for SHAP
SAMPLE_INDEX = 0                         # which test sample to explain
RANDOM_SEED = 42


In [ ]:

# %%
import os
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import load_model
np.random.seed(RANDOM_SEED)
tf.random.set_seed(RANDOM_SEED)

assert os.path.exists(MODEL_PATH), f"Model not found: {MODEL_PATH}"
assert os.path.exists(DATA_PATH), f"Data not found: {DATA_PATH}"

X = np.load(DATA_PATH, allow_pickle=True)
y = np.load(LABELS_PATH, allow_pickle=True) if os.path.exists(LABELS_PATH) else None

# Normalize to shape (N, T, L)
if X.ndim == 2:  # (N, T) -> (N, T, 1)
    X = X[..., None]

N, T, L = X.shape
print(f"X shape: {X.shape} (N={N}, T={T}, L={L})")

model = load_model(MODEL_PATH, compile=False)
try:
    model.compile()
except Exception:
    pass

# Map target class index
if isinstance(CLASS_NAMES, (list, tuple)):
    class_to_idx = {c:i for i,c in enumerate(CLASS_NAMES)}
    target_idx = class_to_idx.get(TARGET_CLASS_NAME, 0)
else:
    target_idx = 0

# Pick sample
idx = int(SAMPLE_INDEX) % N
x_sample = X[idx:idx+1]  # shape (1, T, L)
pred = model.predict(x_sample, verbose=0)
print("Pred logits/softmax:", pred)
print("Predicted class:", np.argmax(pred), "->", (CLASS_NAMES or ["?"])[np.argmax(pred)])

# Choose a background for SHAP (subset of test set)
bg_idx = np.random.choice(N, size=min(BACKGROUND_N, N), replace=False)
background = X[bg_idx]


In [ ]:

# %%
import matplotlib.pyplot as plt
from matplotlib.collections import LineCollection
from matplotlib.colors import TwoSlopeNorm

def plot_ecg_overlay(time, signal, contrib, title="ECG with SHAP/LIME Overlay", lead_name=None, vmin=None, vmax=None):
    """
    Plot a single-lead ECG time series with a color overlay encoding contribution values.

    Args:
        time: (T,) array of time points (seconds or sample index).
        signal: (T,) array of ECG values.
        contrib: (T,) array of contribution scores (positive -> pushes toward class).
        lead_name: optional string, name of the lead.
        vmin, vmax: symmetric color scaling bounds. If None, uses max(|contrib|).
    """
    signal = np.asarray(signal).reshape(-1)
    contrib = np.asarray(contrib).reshape(-1)
    T = len(signal)
    if time is None:
        time = np.arange(T)

    if vmin is None or vmax is None:
        amax = np.nanmax(np.abs(contrib)) if np.nanmax(np.abs(contrib))>0 else 1.0
        vmin, vmax = -amax, amax

    # build colored segments
    points = np.array([time, signal]).T.reshape(-1, 1, 2)
    segments = np.concatenate([points[:-1], points[1:]], axis=1)

    norm = TwoSlopeNorm(vcenter=0.0, vmin=vmin, vmax=vmax)
    lc = LineCollection(segments, cmap='coolwarm', norm=norm)
    lc.set_array(contrib[:-1])
    lc.set_linewidth(1.5)

    fig, ax = plt.subplots(figsize=(10, 3))
    ax.add_collection(lc)
    ax.set_xlim(time.min(), time.max())
    pad = 0.05*(signal.max()-signal.min() + 1e-8)
    ax.set_ylim(signal.min()-pad, signal.max()+pad)
    ax.set_title(title + (f" — Lead: {lead_name}" if lead_name else ""))
    ax.set_xlabel("Time (samples)")
    ax.set_ylabel("mV")
    cbar = plt.colorbar(lc, ax=ax)
    cbar.set_label("Contribution → target class (red = positive, blue = negative)")
    plt.show()

def summarize_top_segments(contrib, segment_len=40, top_k=5, min_gap=20):
    """Return top segments with highest positive contribution sums.
    Args:
        contrib: (T,) contribution vector
        segment_len: window size in timesteps
        top_k: number of segments to report
        min_gap: minimum separation between reported segments
    Returns:
        List of (start, end, score)
    """
    T = len(contrib)
    scores = np.array([contrib[i:i+segment_len].sum() for i in range(0, max(1, T-segment_len+1))])
    idxs = scores.argsort()[::-1]  # descending
    selected = []
    for i in idxs:
        s, e = i, i+segment_len
        if all((abs(s - ps) >= min_gap and abs(e - pe) >= min_gap) for ps, pe, _ in selected):
            selected.append((s, e, scores[i]))
            if len(selected) >= top_k:
                break
    return selected


In [ ]:

# %%
# 2) SHAP: per-timestep, per-lead contributions
import shap

# Choose an explainer based on model type
# For many TF2 Keras models, GradientExplainer works well; try DeepExplainer if supported.
try:
    explainer = shap.GradientExplainer(model, background)
except Exception as e:
    print("GradientExplainer failed, falling back to KernelExplainer. Reason:", e)
    f = lambda data: model.predict(data, verbose=0)
    # Flatten to 2D for KernelExplainer if needed
    X2 = background.reshape((background.shape[0], -1))
    explainer = shap.KernelExplainer(f, X2)

# Compute SHAP values for one sample
try:
    shap_vals = explainer.shap_values(x_sample)  # list per class OR array depending on explainer
except Exception as e:
    print("Explainer direct call failed, trying flattened input. Reason:", e)
    f = lambda data: model.predict(data.reshape((-1, T, L)), verbose=0)
    explainer = shap.KernelExplainer(f, background.reshape((background.shape[0], -1)))
    shap_vals = explainer.shap_values(x_sample.reshape((1, -1)))

# Harmonize SHAP output to shape (num_classes, T, L)
if isinstance(shap_vals, list):
    # list length = num_classes, each (1, T, L)
    shap_arr = np.stack([sv.reshape(T, L) for sv in shap_vals], axis=0)
else:
    # assume (1, T, L) for target only
    shap_arr = shap_vals.reshape(1, T, L)

print("SHAP array shape:", shap_arr.shape)
target_contrib = shap_arr[target_idx]  # (T, L)

# Plot each lead
for lead in range(L):
    plot_ecg_overlay(
        time=np.arange(T),
        signal=x_sample[0, :, lead],
        contrib=target_contrib[:, lead],
        title=f"SHAP overlay toward class '{CLASS_NAMES[target_idx]}'",
        lead_name=f"Lead {lead+1}"
    )

# Summarize suspicious segments for this sample
top_segments = summarize_top_segments(target_contrib.mean(axis=1))
print("Top suspicious segments (by mean contribution across leads):")
for (s, e, sc) in top_segments:
    print(f"  {s:5d}–{e:5d}  score={sc:.3f}")


In [ ]:

# %%
# 3) LIME for 1D time series
from lime.lime_tabular import LimeTabularExplainer
from sklearn.utils import check_random_state

def segment_time_series(x, n_segments=50):
    """Return segment boundaries for 1D segmentation across time."""
    T = x.shape[0]
    bounds = np.linspace(0, T, n_segments+1).astype(int)
    segments = [(bounds[i], bounds[i+1]) for i in range(n_segments)]
    return segments

def lime_explain_timeseries(model, x, class_idx, n_segments=50, num_samples=2000, seed=RANDOM_SEED):
    """
    LIME-like explanation for multi-lead time series using simple 1D segmentation.
    Treats each segment across all leads as a 'feature' that can be masked.
    """
    T, L = x.shape
    segments = segment_time_series(x[:, 0], n_segments=n_segments)
    rng = check_random_state(seed)

    def data_generator(mask_vectors):
        # mask_vectors shape: (M, n_segments), values in {0,1}, 1=keep, 0=mask
        M = mask_vectors.shape[0]
        X_masked = np.repeat(x[None, ...], M, axis=0)  # (M, T, L)
        for m in range(M):
            for j, (s, e) in enumerate(segments):
                if mask_vectors[m, j] == 0:
                    # mask with zeros (or mean); here zeros
                    X_masked[m, s:e, :] = 0.0
        return model.predict(X_masked, verbose=0)

    # Build tabular dataset for LIME with binary 'features' = segments
    n_feat = len(segments)
    training_data = rng.randint(0, 2, size=(min(2048, max(512, n_segments*20)), n_feat))
    explainer = LimeTabularExplainer(
        training_data,
        feature_names=[f"{s}-{e}" for s, e in segments],
        class_names=CLASS_NAMES,
        discretize_continuous=False,
        random_state=rng
    )

    def predict_fn(z):
        # z shape: (M, n_segments) binary masks
        probs = data_generator(z.astype(int))
        return probs

    # Explain around the all-ones (original) mask
    z0 = np.ones((1, n_feat))
    exp = explainer.explain_instance(
        z0[0],
        predict_fn,
        labels=[class_idx],
        num_features=n_feat,
        num_samples=num_samples
    )

    # Build per-timestep contributions by smearing segment weights
    weights = dict(exp.as_list(label=class_idx))
    contrib = np.zeros(T, dtype=float)
    for j, (s, e) in enumerate(segments):
        w = weights.get(f"{s}-{e}", 0.0)
        contrib[s:e] += w / max(1, (e - s))
    return contrib, segments, exp

# Run LIME on our sample
lime_contrib, lime_segments, lime_exp = lime_explain_timeseries(
    model, x_sample[0], class_idx=target_idx, n_segments=60, num_samples=1500
)
print("Computed LIME contribution vector with shape:", lime_contrib.shape)

# Plot a representative lead with LIME overlay
lead_to_plot = 0
plot_ecg_overlay(
    time=np.arange(T),
    signal=x_sample[0, :, lead_to_plot],
    contrib=lime_contrib,  # same contrib applied to each lead; you can extend per-lead masking if desired
    title=f"LIME overlay toward class '{CLASS_NAMES[target_idx]}'",
    lead_name=f"Lead {lead_to_plot+1}"
)


In [ ]:

# %%
# 4) Human-readable summary
def describe_findings(shap_contrib, lime_contrib, top_k=5):
    msg = []
    msg.append(f"Target class: {CLASS_NAMES[target_idx]}")
    # Top leads by total positive contribution
    lead_scores = shap_contrib.clip(min=0).sum(axis=0)  # (L,)
    top_leads = np.argsort(lead_scores)[::-1][:min(top_k, len(lead_scores))]
    msg.append("Top leads by positive SHAP contribution: " + ", ".join([f"Lead {i+1}" for i in top_leads]))

    # Top segments by combined (SHAP mean + LIME)
    comb = shap_contrib.mean(axis=1) + lime_contrib
    segs = summarize_top_segments(comb, segment_len=max(20, T//50), top_k=top_k)
    for rank, (s, e, sc) in enumerate(segs, 1):
        msg.append(f"{rank}. Suspicious interval {s}–{e} (mean+LIME score={sc:.2f})")
    return "\n".join(msg)

summary_text = describe_findings(target_contrib, lime_contrib)
print(summary_text)
